# CXR report labeling pipeline

Driver notebook for sampling → LLM annotate → regex eval → full-dataset regex → LLM verify → final labels.

**You run all cells** (use a GPU machine for SRR-BERT). The agent authors code; it does not execute these jobs in the sandbox.

Environment: `module load conda && conda activate py313`

## 1. Setup

In [1]:
import os
import logging

from medvqa.utils.logging_utils import setup_logging
from medvqa.datasets.cxr_report_labeling.paths import (
    DEFAULT_RUN_ID,
    get_cache_run_dir,
    get_results_run_dir,
    get_srr_bert_cache_dir,
)
from medvqa.datasets.cxr_report_labeling.class_registry import build_class_registry

setup_logging()
logging.getLogger().setLevel(logging.INFO)

RUN_ID = DEFAULT_RUN_ID  # e.g. "v1_k1200_iu400"
CACHE_DIR = get_cache_run_dir(RUN_ID)
RESULTS_DIR = get_results_run_dir(RUN_ID)
SRR_CACHE_DIR = get_srr_bert_cache_dir()  # shared across runs

print("RUN_ID", RUN_ID)
print("CACHE_DIR", CACHE_DIR)
print("RESULTS_DIR", RESULTS_DIR)
print("SRR_CACHE_DIR", SRR_CACHE_DIR)

registry = build_class_registry()
registry.save(os.path.join(RESULTS_DIR, "class_registry.json"))
print("n_classes", len(registry.classes), "regex-backed", len(registry.regex_backed_class_ids()))

PROJECT_ROOT: /home/pamessina/medvqa
DOTENV_PATH: /home/pamessina/medvqa/.env


2026-07-29 12:51:25 | INFO     | root: Logging configured (Color: True).


RUN_ID v1_k1200_iu400
CACHE_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400
RESULTS_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/results/cxr_report_labeling/v1_k1200_iu400
SRR_CACHE_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/srr_bert_leaves
n_classes 55 regex-backed 54


2026-07-29 10:25:23 | INFO     | root: Logging configured (Color: True).


RUN_ID v1_k1200_iu400
CACHE_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400
RESULTS_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/results/cxr_report_labeling/v1_k1200_iu400
SRR_CACHE_DIR /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/srr_bert_leaves
n_classes 55 regex-backed 54


## 2. Phase 0 smoke (constants / registry)

In [2]:
from medvqa.utils.constants import (
    UNIFIED_CXRLT2024_VINDRCXR_CLASSES,
    UNIFIED_CXRLT2024_VINDRCXR_CLASS_TO_REGEX_CLASSES,
)
from medvqa.datasets.regular_expressions.cxr_patterns import _CLASS_NAME_TO_REGEX_PATTERNS

assert "Pulmonary Vascular Congestion" in UNIFIED_CXRLT2024_VINDRCXR_CLASSES
assert "Hilar Congestion" not in UNIFIED_CXRLT2024_VINDRCXR_CLASSES
assert UNIFIED_CXRLT2024_VINDRCXR_CLASS_TO_REGEX_CLASSES["Bone Fracture"] == ["Bone Fracture"]
for c, regs in UNIFIED_CXRLT2024_VINDRCXR_CLASS_TO_REGEX_CLASSES.items():
    if regs is None:
        continue
    for r in regs:
        assert r in _CLASS_NAME_TO_REGEX_PATTERNS, (c, r)
print("Phase 0 OK")

Phase 0 OK


## 3. Load reports (CPU)

For a quick dry-run, set limits (e.g. `IU_LIMIT=100`). For the real pipeline, set all limits to `None`.

In [3]:
from medvqa.datasets.cxr_report_labeling.report_loaders import load_all_datasets

# --- configure ---
MIMIC_LIMIT = None
CHEXPERT_LIMIT = None
REX_LIMIT_PER_SPLIT = None
IU_LIMIT = None
# Example dry-run:
# MIMIC_LIMIT, CHEXPERT_LIMIT, REX_LIMIT_PER_SPLIT, IU_LIMIT = 200, 200, 100, 100

reports_by_dataset = load_all_datasets(
    mimic_limit=MIMIC_LIMIT,
    chexpert_limit=CHEXPERT_LIMIT,
    rex_limit_per_split=REX_LIMIT_PER_SPLIT,
    iu_limit=IU_LIMIT,
)
for ds, rs in reports_by_dataset.items():
    print(ds, len(rs), "example uid", rs[0]["uid"], "chars", len(rs[0]["report_text"]))

2026-07-29 12:51:52 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 227835 MIMIC-CXR reports
2026-07-29 12:51:56 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 223462 CheXpert Plus reports
2026-07-29 12:51:57 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 140000 ReXGradient reports from train
2026-07-29 12:51:58 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 10000 ReXGradient reports from val
2026-07-29 12:51:58 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 10000 ReXGradient reports from test
2026-07-29 12:51:58 | INFO     | medvqa.datasets.cxr_report_labeling.report_loaders: Loaded 3955 IU X-ray reports


mimiccxr 227835 example uid mimiccxr:58829627 chars 407
chexpert_plus 223462 example uid chexpert_plus:patient42142_5 chars 671
rexgradient 160000 example uid rexgradient:pGRDNLZHK1CJMB9DS_aGRDNLD4ATLU63FN8_s1.2.826.0.1.3680043.8.498.98869902613028373370176585666602171978 chars 203
iuxray 3955 example uid iuxray:2509.xml chars 257


mimiccxr 200 example uid mimiccxr:58829627 chars 407
chexpert_plus 200 example uid chexpert_plus:patient42142_5 chars 671
rexgradient 300 example uid rexgradient:pGRDNLZHK1CJMB9DS_aGRDNLD4ATLU63FN8_s1.2.826.0.1.3680043.8.498.98869902613028373370176585666602171978 chars 203
iuxray 100 example uid iuxray:2509.xml chars 257


## 4. Complexity scores (CPU)

In [4]:
import json
from medvqa.datasets.cxr_report_labeling.complexity import score_reports

complexity_by_dataset = {}
for ds, rs in reports_by_dataset.items():
    scores = score_reports(rs)
    complexity_by_dataset[ds] = scores
    path = os.path.join(CACHE_DIR, f"complexity_scores_{ds}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"uids": [r["uid"] for r in rs], "scores": scores}, f)
    print(ds, "mean complexity", sum(scores) / max(len(scores), 1), "->", path)

mimiccxr mean complexity 0.11644391774749271 -> /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400/complexity_scores_mimiccxr.json
chexpert_plus mean complexity 0.7851715280450369 -> /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400/complexity_scores_chexpert_plus.json
rexgradient mean complexity 0.0699375 -> /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400/complexity_scores_rexgradient.json
iuxray mean complexity 0.5130214917825537 -> /mnt/researchers/denis-parra/workspace_pamessina/medvqa/cache/cxr_report_labeling/v1_k1200_iu400/complexity_scores_iuxray.json


## 4b. Inspect complexity ranking (optional)

Interactive sanity check: browse reports ranked by complexity within each loaded dataset. Use **highest** to confirm top K1 candidates look dense/rare-token-heavy; flip to **lowest** for contrast.

Requires `ipywidgets` (available in `py313`). Re-run after changing load limits or recomputing scores.

In [5]:
from medvqa.datasets.cxr_report_labeling.visualization import inspect_complexity_interactively

# Set max_report_chars=2000 to truncate long CheXpert+/MIMIC reports in the widget output.
inspect_complexity_interactively(
    reports_by_dataset,
    complexity_by_dataset,
    max_report_chars=None,
)

Output()

## 5. SRR-BERT-Leaves (GPU)

Label all loaded reports with sentence-split + OR-merge. Results are cached under `SRR_CACHE_DIR` by report text hash.

**Requires GPU** for full corpora. You can also run:
`python -m medvqa.scripts.cxr_report_labeling.run_srr_bert_leaves`

In [10]:
x = """
Hyperinflated lung
Edema
Reticular interstitial pattern
Vascular redistribution
Scoliosis
Pulmonary fibrosis
Subcutaneous emphysema
Ground glass pattern
Kerley lines
Pneumonia
Bronchiectasis
Tubes
Degenerative changes
Cardiomegaly
Aortic elongation
Consolidation
Lines
Flattened diaphragm
Pneumothorax
Bulla
Mediastinal enlargement
Pleural thickening
Hilar enlargement
Pleural effusion
Hemidiaphragm elevation
Abnormal foreign body
Mediastinal mass
Atelectasis
Sternotomy
Hiatal hernia
Lung Mass
Osteopenia
Aortic endoprosthesis
Pleural mass
Cavitation
Nodule
Pacemaker
Aortic atheromatosis
Costophrenic angle blunting
Bone lesion
Calcified heart valve
Fracture
Rib fracture
Artificial heart valve
Pleural plaques
Granuloma
Calcified granuloma"""

for y in sorted(x.split('\n')):
    if y.strip():
        print(y.strip())

Abnormal foreign body
Aortic atheromatosis
Aortic elongation
Aortic endoprosthesis
Artificial heart valve
Atelectasis
Bone lesion
Bronchiectasis
Bulla
Calcified granuloma
Calcified heart valve
Cardiomegaly
Cavitation
Consolidation
Costophrenic angle blunting
Degenerative changes
Edema
Flattened diaphragm
Fracture
Granuloma
Ground glass pattern
Hemidiaphragm elevation
Hiatal hernia
Hilar enlargement
Hyperinflated lung
Kerley lines
Lines
Lung Mass
Mediastinal enlargement
Mediastinal mass
Nodule
Osteopenia
Pacemaker
Pleural effusion
Pleural mass
Pleural plaques
Pleural thickening
Pneumonia
Pneumothorax
Pulmonary fibrosis
Reticular interstitial pattern
Rib fracture
Scoliosis
Sternotomy
Subcutaneous emphysema
Tubes
Vascular redistribution


In [ ]:
import torch
from medvqa.datasets.cxr_report_labeling.schemas import report_text_hash
from medvqa.datasets.cxr_report_labeling.srr_bert_leaves import (
    SRRBertLeavesLabeler,
    SRR_BERT_LEAVES_CLASS_NAMES,
    labels_to_names,
)

print("cuda available", torch.cuda.is_available())
labeler = SRRBertLeavesLabeler(
    device="cuda" if torch.cuda.is_available() else "cpu",
    default_batch_size=32,
    cache_dir=SRR_CACHE_DIR,
    verbose=True,
)

srr_vectors_by_dataset = {}
for ds, rs in reports_by_dataset.items():
    texts = [r["report_text"] for r in rs]
    hashes = [report_text_hash(t) for t in texts]
    vecs = labeler.get_labels_for_reports(texts, report_hashes=hashes, batch_size=32, save_every=2000)
    srr_vectors_by_dataset[ds] = vecs
    print(ds, "done", len(vecs), "example labels", labels_to_names(vecs[0])[:10])
labeler.save_caches()
print("SRR cache saved to", SRR_CACHE_DIR)

## 6–7. Sample K1/K2/K3 → `samples.jsonl`

Uses regex strata + SRR strata (from cache) + complexity. Defaults: MIMIC/ReX/CheXpert+ 200/800/200, IU 50/250/100.

In [ ]:
from medvqa.scripts.cxr_report_labeling.sample_reports import run_sampling

samples_path = run_sampling(
    run_id=RUN_ID,
    seed=0,
    num_regex_processes=4,  # increase if helpful
    require_srr=True,       # set False only if you intentionally skip SRR
    limits={
        "mimiccxr": MIMIC_LIMIT,
        "chexpert_plus": CHEXPERT_LIMIT,
        "rexgradient": REX_LIMIT_PER_SPLIT,
        "iuxray": IU_LIMIT,
    },
)
print("samples_path", samples_path)

In [ ]:
from collections import Counter
from medvqa.utils.files_utils import load_jsonl

samples = load_jsonl(samples_path)
print("n_samples", len(samples))
print("by dataset", Counter(s["dataset"] for s in samples))
print("by method", Counter(s["sampling"]["method"] for s in samples))
print("example", samples[0]["uid"], samples[0]["sampling"])

## 8. LLM annotate samples (API — capped)

Set `MAX_QUERIES` small for a pilot, inspect JSONL, then increase.
Resume keys are `(uid, class_id, prompt_hash, system_prompt_hash, model_name)`.

In [ ]:
from medvqa.scripts.cxr_report_labeling.annotate_reports_with_llm import run_annotation

MODEL_NAME = "gemini-2.5-flash-lite-preview-09-2025"
API_KEY_NAME = "GOOGLE_API_KEY"  # or GEMINI_API_KEY — must match your .env
MAX_QUERIES = 20  # raise when ready

info = run_annotation(
    run_id=RUN_ID,
    samples_path=samples_path,
    model_name=MODEL_NAME,
    api_key_name=API_KEY_NAME,
    api_type="gemini",
    max_queries=MAX_QUERIES,
    # class_ids=["Cardiomegaly"],  # optional filter
    estimate_only=True,  # flip to False to spend money
    dry_run=True,
)
info

In [ ]:
# Uncomment to actually call the API for this batch:
# info = run_annotation(
#     run_id=RUN_ID,
#     samples_path=samples_path,
#     model_name=MODEL_NAME,
#     api_key_name=API_KEY_NAME,
#     api_type="gemini",
#     max_queries=MAX_QUERIES,
# )
# info

## 9. Eval regex vs LLM + FP/FN dumps

In [ ]:
from medvqa.scripts.cxr_report_labeling.eval_regex_vs_llm import evaluate_regex_vs_llm

ANN_PATH = os.path.join(
    CACHE_DIR, "llm_annotations", MODEL_NAME.replace("/", "_"), "sample_annotations.jsonl"
)
print("annotations exist?", os.path.exists(ANN_PATH), ANN_PATH)

if os.path.exists(ANN_PATH):
    eval_out = evaluate_regex_vs_llm(
        run_id=RUN_ID,
        samples_path=samples_path,
        annotations_path=ANN_PATH,
        backup=True,  # copies cxr_classes into .agent/
    )
    metrics = eval_out["metrics"]
    print("eval_dir", eval_out["eval_dir"])
    # show worst recall classes
    worst = sorted(metrics.items(), key=lambda kv: kv[1]["recall"])[:15]
    for c, m in worst:
        print(f"{c:40s} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} n={m['n_annotated']}")
else:
    print("Skip eval until sample annotations exist")

## 10. Visualization (P/R/F1 + FP/FN browser)

In [ ]:
from medvqa.datasets.cxr_report_labeling.visualization import plot_prf1_bars, show_fp_fn_example
from medvqa.utils.files_utils import load_jsonl

if "metrics" in dir() and metrics:
    plot_prf1_bars(metrics)

    # Browse false negatives for a class
    CLASS = "Cardiomegaly"
    stem = CLASS.lower().replace("/", "_").replace(" ", "_")
    fn_path = os.path.join(CACHE_DIR, "regex_eval", f"{stem}_fn.jsonl")
    fp_path = os.path.join(CACHE_DIR, "regex_eval", f"{stem}_fp.jsonl")
    if os.path.exists(fn_path):
        fns = load_jsonl(fn_path)
        print("n FN", len(fns))
        if fns:
            show_fp_fn_example(fns[0])
    if os.path.exists(fp_path):
        fps = load_jsonl(fp_path)
        print("n FP", len(fps))
        if fps:
            show_fp_fn_example(fps[0])

## 11. Full-dataset regex apply

Run once per dataset after regex refinement.

In [ ]:
from medvqa.scripts.cxr_report_labeling.apply_regex_to_dataset import apply_regex_to_dataset

# Example: IU first (small)
# path = apply_regex_to_dataset(dataset="iuxray", run_id=RUN_ID, num_processes=4, include_spans=False)
# print(path)

# for ds in ["iuxray", "mimiccxr", "chexpert_plus", "rexgradient"]:
#     print(apply_regex_to_dataset(dataset=ds, run_id=RUN_ID, num_processes=4))

## 12. LLM verify regex positives (API — capped)

In [ ]:
from medvqa.scripts.cxr_report_labeling.verify_positive_matches_with_llm import run_verify_positives

# info = run_verify_positives(
#     dataset="iuxray",
#     run_id=RUN_ID,
#     model_name=MODEL_NAME,
#     api_key_name=API_KEY_NAME,
#     max_queries=20,
#     estimate_only=True,
# )
# info

## 13. Materialize final labels

Regex miss → `Unmentioned`; regex hit → LLM 5-way. Normal annotated last (Support-Devices-only still Normal-eligible).

In [ ]:
from medvqa.scripts.cxr_report_labeling.materialize_final_labels import materialize_final_labels

# path = materialize_final_labels(dataset="iuxray", run_id=RUN_ID, model_name=MODEL_NAME)
# print(path)

## Ontology invalidation (optional)

After radiologist-driven class/prompt changes, diff registries to see what to re-run.

In [ ]:
from medvqa.datasets.cxr_report_labeling.class_registry import ClassRegistry, build_class_registry
from medvqa.datasets.cxr_report_labeling.invalidation import diff_registries

old_path = os.path.join(RESULTS_DIR, "class_registry.json")
if os.path.exists(old_path):
    old = ClassRegistry.load(old_path)
    new = build_class_registry()  # from current prompts/constants
    report = diff_registries(old, new)
    print(report.summary())
else:
    print("No saved registry yet")